# CTC — Connectionist Temporal Classification (2006)

_Papers · Sequence & NLP_

**Train a recurrent network to map an unsegmented input sequence to a shorter label string with no frame-by-frame labels: add a blank symbol, collapse repeats-and-blanks, and sum over every valid alignment with a forward-backward dynamic program.**

---

This notebook is a practice scaffold for the **CTC — Connectionist Temporal Classification (2006)** lesson. We build the forward (alpha) recursion by hand, check it against PyTorch's `CTCLoss`, then ablate the repeat-skip rule to see how it keeps a blank between two identical labels. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — The extended label and the forward recursion

CTC inserts a **blank** symbol between every label and at both ends, giving the extended sequence `l' = blank, l1, blank, l2, ...` of length `2|l|+1`. The forward variable `alpha[t,s]` is the total probability of all alignments that have produced the first `s` positions of `l'` by frame `t`. Each step can **stay**, **advance one**, or **skip one** — but the skip is forbidden when the current symbol is blank or equals the symbol two positions back (Eq. 6), which is what keeps repeated labels apart.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

np.set_printoptions(precision=4, suppress=True)
BLANK = 0   # PyTorch CTCLoss convention: blank is index 0

def extend(labels, blank=BLANK):
    """l' = blank, l1, blank, l2, blank, ... ; length 2*len(labels)+1."""
    lp = [blank]
    for c in labels:
        lp += [c, blank]
    return lp

def ctc_forward(y, labels, blank=BLANK, allow_repeat_skip=True):
    """CTC forward (alpha) recursion. y: (T,K) softmax rows. Returns (p, alpha)."""
    T, K = y.shape
    lp = extend(labels, blank)
    S = len(lp)
    a = np.zeros((T, S))
    a[0, 0] = y[0, lp[0]]                 # start in leading blank
    a[0, 1] = y[0, lp[1]]                 # or on the first label
    for t in range(1, T):
        for s in range(S):
            v = a[t-1, s]                                  # stay
            if s - 1 >= 0:
                v += a[t-1, s-1]                           # advance one
            # skip term: allowed unless l'_s is blank, or l'_s == l'_{s-2}
            forbids = (lp[s] == blank) or (allow_repeat_skip and lp[s] == lp[s-2])
            if s - 2 >= 0 and not forbids:
                v += a[t-1, s-2]                           # skip a position
            a[t, s] = v * y[t, lp[s]]
    p = a[T-1, S-1] + a[T-1, S-2]         # Eq. 8: end on last label or trailing blank
    return p, a

### Step 2 — Run a worked example for target "ab"

Take four frames over three symbols (blank, a, b) and target `"ab"`, so `l' = [0,1,0,2,0]`. The forward pass fills the `alpha` matrix; the probability of the label is the sum of the two bottom-right entries (ending on the last label or the trailing blank), and the loss is `-ln p`. We also spot-check one skip step, `alpha[2,3]`, by hand.

In [ ]:
# Worked example: target "ab", T=4 frames, K=3 (blank, a, b).
y = np.array([[0.10, 0.60, 0.30],
              [0.20, 0.50, 0.30],
              [0.30, 0.30, 0.40],
              [0.40, 0.20, 0.40]])
labels = [1, 2]                            # "a","b"   (l' = [0,1,0,2,0])

p, alpha = ctc_forward(y, labels)
loss_mine = -np.log(p)

print("l' =", extend(labels))
print("alpha matrix (rows=t, cols=s):\n", alpha)
print("worked skip step alpha[2,3] = (0.18+0.12+0.35)*0.40 =", round(alpha[2, 3], 4))  # 0.26
print("p(l|x) =", round(p, 4), "  loss -ln p =", round(loss_mine, 6))                  # 0.3304 / 1.107451

### Step 3 — Verify against PyTorch's CTCLoss

The from-scratch result should match `torch.nn.CTCLoss`. Torch expects **log**-probabilities of shape `(T, N, C)`, so we log the emission matrix and add a batch dimension. The two losses should agree to numerical precision.

In [ ]:
# Verify against torch.nn.CTCLoss (it expects LOG-probs, shape (T, N, C)).
log_probs = torch.log(torch.tensor(y, dtype=torch.double)).unsqueeze(1)   # (T,1,K)
targets = torch.tensor([labels], dtype=torch.long)                        # (1,L)

ctc = nn.CTCLoss(blank=BLANK, reduction='none')
input_lengths = torch.tensor([y.shape[0]])
target_lengths = torch.tensor([len(labels)])
loss_torch = ctc(log_probs, targets, input_lengths, target_lengths).item()

print("loss torch =", round(loss_torch, 6),
      "  allclose:", np.allclose(loss_mine, loss_torch))                   # True

### Step 4 — Ablation: the repeat-skip rule on target "aa"

For a repeated target `"aa"`, the extended sequence is `[0,1,0,1,0]` — a blank separates the two a's. The rule `l'_s == l'_{s-2}` forbids skipping over that blank; drop it and alignments that collapse to a single `"a"` get counted, inflating `p(l|x)` and wrongly **lowering** the loss. Torch agrees with the correct value, confirming the rule matters.

In [ ]:
# ABLATION: the l'_s == l'_{s-2} skip rule, on the repeated target "aa".
y2 = np.array([[0.1, 0.7, 0.2],
               [0.5, 0.4, 0.1],
               [0.1, 0.6, 0.3],
               [0.4, 0.5, 0.1]])
aa = [1, 1]                                # "aa" -> l' = [0,1,0,1,0]; middle blank separates the a's

p_correct, _ = ctc_forward(y2, aa, allow_repeat_skip=True)   # rule ON  (correct CTC)
p_buggy,   _ = ctc_forward(y2, aa, allow_repeat_skip=False)  # rule OFF (skip the separating blank)

print("ABLATION on target 'aa':")
print("  correct loss (skip forbidden between equal labels):", round(-np.log(p_correct), 4))  # 1.5028
print("  buggy   loss (skip allowed -> 'aa' leaks to 'a')  :", round(-np.log(p_buggy),   4))  # 0.4024

# torch agrees with the CORRECT value:
lp2 = torch.log(torch.tensor(y2, dtype=torch.double)).unsqueeze(1)
lt2 = nn.CTCLoss(blank=BLANK, reduction='none')(
        lp2, torch.tensor([aa]), torch.tensor([4]), torch.tensor([2])).item()
print("  torch loss:", round(lt2, 4), " matches correct:", np.allclose(-np.log(p_correct), lt2))
print("Dropping the rule wrongly lowers the loss -> it counts paths that collapse to 'a', not 'aa'.")
print("(Tiny deterministic run, not the paper's TIMIT numbers.)")

## Visualize the data & results

_On the repeated-label target 'aa', what does the CTC loss do if you drop the l'_s = l'_{s-2} skip rule that keeps a blank between the two a's?_

### Step 1 — A compact forward pass for the ablation panel

This panel is self-contained, so we redefine `extend` and `ctc_forward` in a trimmed form that returns just the label probability. The `allow_repeat_skip` flag is the only knob: it switches the `l'_s == l'_{s-2}` guard on or off.

In [ ]:
import numpy as np

BLANK = 0

def extend(labels):
    lp = [BLANK]
    for c in labels:
        lp += [c, BLANK]
    return lp

def ctc_forward(y, labels, allow_repeat_skip=True):
    T = y.shape[0]
    lp = extend(labels)
    S = len(lp)
    a = np.zeros((T, S))
    a[0, 0] = y[0, lp[0]]
    a[0, 1] = y[0, lp[1]]
    for t in range(1, T):
        for s in range(S):
            v = a[t-1, s]
            if s - 1 >= 0:
                v += a[t-1, s-1]
            forbids = (lp[s] == BLANK) or (allow_repeat_skip and lp[s] == lp[s-2])
            if s - 2 >= 0 and not forbids:
                v += a[t-1, s-2]
            a[t, s] = v * y[t, lp[s]]
    return a[T-1, S-1] + a[T-1, S-2]

### Step 2 — Compare correct vs buggy loss on "aa"

Run the same emissions both ways. With the guard on, the loss is the correct `1.5028` (matching torch); with it off, the path skips the separating blank, `p(l|x)` inflates, and the loss drops to a bogus `0.4024`.

In [ ]:
y = np.array([[0.1, 0.7, 0.2],
              [0.5, 0.4, 0.1],
              [0.1, 0.6, 0.3],
              [0.4, 0.5, 0.1]])
aa = [1, 1]                                       # "aa" -> l' = [0,1,0,1,0]

correct = -np.log(ctc_forward(y, aa, allow_repeat_skip=True))   # 1.5028 (= torch)
buggy   = -np.log(ctc_forward(y, aa, allow_repeat_skip=False))  # 0.4024 (wrong)
print("correct loss:", round(correct, 4))
print("buggy   loss:", round(buggy,   4))
# Dropping the l'_s == l'_{s-2} guard lets the path skip the blank between the two a's,
# so it counts paths that collapse to 'a' -> p(l|x) inflated -> loss wrongly low.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The skip-rule ablation. For the repeated target "aa", explain why the skip from $s-2$ to
            $s$ must be forbidden, and what the loss does if you (wrongly) allow it. Then state which value the
            forward pass should report.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Extend "aa" $=[1,1]$ to $l'=[0,1,0,1,0]$ (blank, a, blank, a, blank). The middle blank at $s=2$ separates the two a's. — _$\mathcal{B}$ merges adjacent identical labels, so without that blank the two a's collapse into one._
- At $s=3$ (the second 'a'), $l'_3=$'a' and $l'_1=$'a' are identical, so Eq. 6 forbids the $\alpha_{t-1}(1)$ skip term. Only stay ($s{=}3$) and advance ($s{=}2$) feed in. — _Skipping from the first 'a' straight to the second would erase the separating blank and produce "a", not "aa"._
- Allowing the skip anyway adds invalid paths, raising $p(l|x)$ and so lowering the loss &mdash; in our run from the correct $1.503$ down to a bogus $0.402$. — _Those extra paths actually collapse to "a"; counting them is double-counting a different labelling._

**Answer:** The skip from $s-2$ is forbidden when $l'_{s-2}=l'_s$ because it would jump over the blank
                 that keeps two identical labels apart, merging "aa" into "a". The correct CTC loss on our
                 tiny example is $\mathbf{1.503}$ (and torch.nn.CTCLoss agrees,
                 allclose True). The buggy version that allows the skip reports a wrongly low
                 $\approx0.402$. The CODEVIZ panel plots this gap.

</details>

**Problem 2.** Recompute the worked-example skip step by hand. Using row $t=1$ of $\alpha$,
            $[0.02,\,0.35,\,0.12,\,0.18,\,0]$, and $y^2 = [0.30,0.30,0.40]$ (cols $[b,a,b]$), compute
            $\alpha_2(3)$ for the target "ab" (so $l'=[0,1,0,2,0]$).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Position $s=3$ holds symbol $l'_3=2$ ('b'); $l'_1=1$ ('a'). They differ and neither is blank, so the skip is allowed. — _Eq. 6 permits the $s-2$ term unless $l'_s$ is blank or $l'_{s-2}=l'_s$ &mdash; here neither holds._
- Sum the three predecessors: $\alpha_1(3)+\alpha_1(2)+\alpha_1(1) = 0.18+0.12+0.35 = 0.65$. — _Stay ($s{=}3$), advance ($s{=}2$), and the legal skip ($s{=}1$) all feed position $3$._
- Multiply by the emission $y^2_{l'_3}=y^2_b=0.40$: $0.65\times0.40 = 0.26$. — _Every $\alpha$ entry charges the probability of emitting its symbol at the current frame._

**Answer:** $\alpha_2(3) = (0.18+0.12+0.35)\times0.40 = 0.65\times0.40 = \mathbf{0.26}$ &mdash;
                 exactly the value in the worked-example matrix, and the notebook reproduces it.

</details>

**Problem 3.** Why does CTC sum over all paths in $\mathcal{B}^{-1}(l)$ (Eq. 3) instead of picking the
            single most likely alignment? What would training on the single best path lose?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The label string $l$ never tells us which frame is which label, so there is no \"true\" alignment to single out. — _The data is unsegmented &mdash; that is the whole problem CTC solves._
- Summing over every valid alignment makes $p(l|x)$ the actual probability of the string, marginalising out the unknown alignment. — _Marginalisation is the principled way to handle a latent variable (here, the alignment) we do not observe._
- Training on only the single best path would commit early to one alignment and ignore the probability mass in all the others, giving a biased, lower-variance-but-wrong objective. — _Early hard alignment can lock onto a poor segmentation; the soft sum lets the network discover alignments itself._

**Answer:** Because the data is unsegmented, no single alignment is \"the\" right one. Eq. 3
                 marginalises over the latent alignment by summing every path that collapses to $l$, giving the
                 true string probability $p(l|x)$; the forward-backward DP computes that sum in $O(T|l'|)$. Using
                 only the best path would throw away the mass in all other valid alignments and bias training
                 toward a prematurely chosen segmentation.

</details>